# 🤖 AI แชทสำหรับ AIoT Cucumber Classroom (บทที่ 6)

โน้ตบุ๊กนี้เปิด **LLM ขนาดเล็ก (Ollama)** บน Colab แล้วสร้างลิงก์ให้ gateway ของคุณเรียกใช้ ไม่ต้องมีการ์ดจอเอง ไม่ต้องสมัครอะไรเพิ่ม

### ทำตามนี้ (ประมาณ 5–8 นาที ครั้งแรก)
1. เมนู **Runtime → Change runtime type → T4 GPU** (ไม่เลือกก็ได้ แต่ช้ามาก)
2. กด **Runtime → Run all** หรือกดเล่นทีละเซลล์จากบนลงล่าง
3. เซลล์ที่ 5 จะพิมพ์บรรทัด `OLLAMA_URL=...` ให้ **คัดลอกไปใส่ `gateway/.env`** แล้วสั่ง `docker compose up -d mqtt_collector`
4. **อย่าปิดแท็บ** และปล่อยเซลล์ที่ 5 รันค้างไว้ตลอดการใช้งาน

> ลองเล่น: เซลล์ที่ 4 ให้เปลี่ยนคำถาม ค่าเซนเซอร์ และ system prompt ได้ ดูว่าคำตอบเปลี่ยนอย่างไร

> ขั้นตอนทั้งหมดมีที่ [หน้าเว็บบทที่ 6](https://arefandywae-lab.github.io/aiot-cucumber-classroom/lessons/6-ai-chat.html)

In [ ]:
# 1) ตรวจว่าได้ GPU ไหม แล้วติดตั้ง Ollama
import subprocess, shutil
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "⚠ ไม่เจอ GPU: เลือก Runtime → Change runtime type → T4 GPU ก่อน (ใช้ CPU ได้แต่ช้า)")
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
print("ollama:", shutil.which("ollama") or "ติดตั้งไม่สำเร็จ")

In [ ]:
# 2) เปิดเซิร์ฟเวอร์ Ollama และดาวน์โหลดโมเดล (ครั้งแรกใช้เวลาสักครู่)
MODEL = "qwen2.5:3b"  #@param ["qwen2.5:3b", "qwen2.5:1.5b", "gemma2:2b", "llama3.2:3b"]
import subprocess, time, urllib.request
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags"); break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("ollama serve ไม่ตอบ ลองรันเซลล์ 1 ใหม่")
!ollama pull {MODEL}

In [ ]:
# 3) ทดสอบว่าโมเดลตอบได้ (ใช้ prompt แบบเดียวกับที่ gateway ส่ง)
import json, urllib.request

def ask_llm(question, context, system, model=MODEL):
    payload = {"model": model, "stream": False, "messages": [
        {"role": "system", "content": system},
        {"role": "user", "content": f"ข้อมูลเซนเซอร์ล่าสุด:\n{context}\n\nคำถาม: {question}"}]}
    req = urllib.request.Request("http://localhost:11434/api/chat", json.dumps(payload).encode(),
                                 {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=120))["message"]["content"].strip()

SYSTEM = ("คุณคือผู้ช่วยสอนวิทยาศาสตร์สำหรับนักเรียน ม.ปลาย ตอบเป็นภาษาไทย สั้น กระชับ เข้าใจง่าย "
          "ใช้เฉพาะข้อมูลเซนเซอร์ที่ให้มาเท่านั้น ถ้าข้อมูลไม่พอให้บอกตามตรง ห้ามเดาตัวเลข")
print(ask_llm("ตอนนี้ร้อนไหม", "- อุณหภูมิ HTS221: 33.1 °C\n- ความชื้น: 62 %", SYSTEM))

## 🧪 ลองเล่น: เปลี่ยนแล้วดูว่าคำตอบเปลี่ยนไหม
ลองถาม **คำถามที่ข้อมูลตอบไม่ได้** (เช่น "พรุ่งนี้ฝนตกไหม") แล้วดูว่าโมเดล *ยอมรับว่าไม่รู้* หรือ *เดามั่ว*
จากนั้นแก้ `system prompt` ให้เข้มขึ้นหรือหลวมลง แล้วเทียบกัน

In [ ]:
#@title ลองเล่น: คำถาม ค่าเซนเซอร์ และ system prompt
QUESTION = "พรุ่งนี้ฝนตกไหม"  #@param {type:"string"}
TEMPERATURE = 33.1  #@param {type:"number"}
HUMIDITY = 62  #@param {type:"slider", min:10, max:100, step:1}
PRESSURE = 1004  #@param {type:"number"}
ANOMALY = False  #@param {type:"boolean"}
USE_MY_SYSTEM_PROMPT = False  #@param {type:"boolean"}
MY_SYSTEM_PROMPT = "ตอบสั้นๆ เป็นภาษาไทย ถ้าข้อมูลไม่พอให้บอกว่า ไม่ทราบ"  #@param {type:"string"}

context = (f"- อุณหภูมิ HTS221: {TEMPERATURE} °C\n- ความชื้น: {HUMIDITY} %\n- ความดันอากาศ: {PRESSURE} hPa\n"
           + ("- ความผิดปกติที่ตรวจพบ: อุณหภูมิสูงขึ้น (z=5.2)" if ANOMALY else "- ยังไม่พบความผิดปกติ"))
print("คำตอบ:", ask_llm(QUESTION, context, MY_SYSTEM_PROMPT if USE_MY_SYSTEM_PROMPT else SYSTEM))

In [ ]:
# 5) สร้างลิงก์สาธารณะด้วย cloudflared (ไม่ต้องสมัครบัญชี) แล้วรันค้างไว้
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import re, subprocess, threading
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434",
                           "--http-host-header", "localhost:11434"],
                          stderr=subprocess.PIPE, text=True)
found = threading.Event()
def watch():                       # อ่าน log ต่อเนื่อง ไม่งั้น pipe เต็มแล้ว tunnel ค้าง
    for line in tunnel.stderr:
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if m and not found.is_set():
            url = m.group(0); found.set()
            print("\n✅ พร้อมแล้ว คัดลอกสองบรรทัดนี้ไปใส่ gateway/.env\n")
            print(f"OLLAMA_URL={url}\nOLLAMA_MODEL={MODEL}\n")
            print("แล้วสั่ง:  docker compose up -d mqtt_collector")
            print(f"ทดสอบจากเครื่องคุณ:  curl {url}/api/tags")
threading.Thread(target=watch, daemon=True).start()
found.wait(60) or print("⚠ ยังไม่ได้ลิงก์ ลองหยุดเซลล์นี้แล้วรันใหม่")
import time
while True:                        # ปล่อยเซลล์นี้ให้รันค้างไว้
    time.sleep(60)

### หมายเหตุ
- ลิงก์เปลี่ยนทุกครั้งที่รันใหม่ และ Colab ฟรีจะหยุดเองเมื่อไม่มีการใช้งานหรือเกินเวลา (ประมาณ 90 นาที ถึง 12 ชั่วโมง)
- ถ้า gateway ติดต่อ LLM ไม่ได้ ระบบจะตอบด้วยกฎง่ายๆ แทนโดยอัตโนมัติ (ไม่พัง)
- **ใครมีลิงก์ก็ใช้โมเดลของคุณได้** อย่าแชร์ต่อ และข้อความที่ถามจะผ่านเซิร์ฟเวอร์ของ Google/Cloudflare อย่าพิมพ์ข้อมูลส่วนตัว
- ข้อความ `Address already in use`: เซลล์ 2 รันซ้ำ ให้ Runtime → Restart session แล้วรันใหม่ตั้งแต่ต้น